In [7]:
import pandas as pd

df_logs = pd.read_parquet("../logs/predictions_log.parquet")


In [ ]:
df_logs.head()

,num__SK_ID_CURR,num__BUREAU_SK_ID_BUREAU_max,num__POS_SK_DPD_min,num__CC_SK_DPD_DEF_max,num__PREV_NAME_TYPE_SUITE_Family,num__PREV_DAYS_FIRST_DRAWING_sum,num__PREV_NAME_GOODS_CATEGORY_Other,num__DAYS_LAST_PHONE_CHANGE,num__CC_CNT_DRAWINGS_OTHER_CURRENT_mean,num__PREV_DAYS_FIRST_DRAWING_max,...,cpu_percent,ram_percent,system_load,num_threads,method,status,error_message,latency_ms,request_size_bytes,response_size_bytes
0,-0.302429,0.326039,0.249338,NaN,-0.535936,-0.300165,0.23674,-0.987792,NaN,0.246419,...,14.5,78.3,0.0,21,POST,success,NaN,383.741379,12798.0,0.0
1,-0.147132,0.553947,0.249338,NaN,-0.535936,0.742804,0.23674,0.895050,NaN,0.246419,...,15.8,78.3,0.0,21,POST,success,NaN,102.715015,12775.0,0.0
2,-0.429638,0.448635,0.249338,NaN,0.494736,-0.821649,0.23674,0.962033,NaN,0.246419,...,12.2,78.4,0.0,21,POST,success,NaN,67.815304,12875.0,0.0
3,0.246223,0.491943,0.249338,0.691535,2.556081,4.392488,0.23674,-2.192274,1.997573,0.246419,...,12.4,78.4,0.0,21,POST,success,NaN,89.750528,12927.0,0.0
4,-1.169600,0.699197,0.249338,NaN,-0.535936,-0.821649,0.23674,0.744033,NaN,0.246419,...,16.8,78.3,0.0,21,POST,success,NaN,68.776608,12836.0,0.0


In [19]:
df_logs['inference_ms']

0     NaN
1     NaN
2     NaN
3     NaN
4     NaN
       ..
195   NaN
196   NaN
197   NaN
198   NaN
199   NaN
Name: inference_ms, Length: 200, dtype: float64

In [9]:
df_logs["latency_ms"].describe()

count    200.000000
mean      86.127709
std       29.946769
min       58.841944
25%       70.760190
50%       76.784968
75%       94.737291
max      383.741379
Name: latency_ms, dtype: float64

In [13]:
df_logs["score_x"].describe()

count    200.000000
mean       0.213953
std        0.161526
min        0.021356
25%        0.084569
50%        0.191494
75%        0.286569
max        0.793830
Name: score_x, dtype: float64

In [17]:
lat_mean = df_logs["latency_ms"].mean()
lat_p95 = df_logs["latency_ms"].quantile(0.95)
lat_max = df_logs["latency_ms"].max()

cpu_mean = df_logs["cpu_percent"].mean()
ram_mean = df_logs["ram_percent"].mean()

infer_mean = df_logs["inference_ms"].mean()
infer_p95 = df_logs["inference_ms"].quantile(0.95)

print(f"latency_mean_p95_max: {lat_mean, lat_p95, lat_max}")
print(f"cpu_ram: {cpu_mean, ram_mean}")
print(f"infer: {infer_p95, infer_mean}")

latency_mean_p95_max: (86.12770915031433, 118.6904191970825, 383.7413787841797)
cpu_ram: (23.789499999999997, 73.7385)
infer: (nan, nan)


In [11]:
df_logs["decision_x"].value_counts()

decision_x
ACCORDÉ    183
REFUSÉ      17
Name: count, dtype: int64

In [16]:
import cProfile
import pstats
import joblib
import pandas as pd

pipe = joblib.load("../BestModel/pipeline_complet.joblib")
df = joblib.load("../data/app_test_clean_v2.joblib").head(1)

def benchmark():
    for _ in range(100):
        pipe.predict_proba(df)

cProfile.run(
    "benchmark()",
    "profiling.prof")

stats = pstats.Stats("profiling.prof")
stats.sort_stats("cumtime")
stats.print_stats(18)

c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.4.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TargetEncoder from version 1.4.2 when using version 

Sun Jun  7 20:34:13 2026    profiling.prof

         3120174 function calls (3086873 primitive calls) in 6.680 seconds

   Ordered by: cumulative time
   List reduced from 703 to 18 due to restriction <18>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    6.693    6.693 {built-in method builtins.exec}
        1    0.000    0.000    6.693    6.693 <string>:1(<module>)
        1    0.001    0.001    6.693    6.693 C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12576\161287911.py:9(benchmark)
      100    0.002    0.000    6.690    0.067 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\pipeline.py:815(predict_proba)
  500/100    0.010    0.000    6.530    0.065 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\utils\_set_output.py:314(wrapped)
      100    0.037    0.000    6.521    0.065 c:\Users\Lenovo\anaconda3\envs\mlops\Lib\site-packages\sklearn\compose\_column_transformer.py:1031(transform)
      100    0

In [23]:
import time

try:
    df_example = joblib.load("../data/app_test_clean_v2.joblib")
except:
    df_example = None

X = df_example.iloc[[0]]

t0 = time.perf_counter()

X_transformed = pipe[:-1].transform(X)

t1 = time.perf_counter()

score = pipe[-1].predict_proba(X_transformed)

t2 = time.perf_counter()

print("preprocessing :", (t1 - t0)*1000)
print("model :", (t2 - t1)*1000)
print("total :", (t2 - t0)*1000)

preprocessing : 37.56080009043217
model : 0.9570997208356857
total : 38.51789981126785


In [ ]:
start = time.perf_counter()

for i in range(100):
    pipe.predict_proba(df_example.iloc[[i]])

end = time.perf_counter()

print(
    "100 appels unitaires :",
    (end - start) * 1000,
    "ms"
)

100 appels unitaires : 3832.566499710083 ms


In [22]:
X_batch = df_example.iloc[:100]

start = time.perf_counter()

pipe.predict_proba(X_batch)

end = time.perf_counter()

print(
    "1 batch de 100 lignes :",
    (end - start) * 1000,
    "ms"
)

1 batch de 100 lignes : 63.72349988669157 ms


In [25]:
df_example = joblib.load("../data/app_test_clean_v2.joblib")
df_example.columns

Index(['SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR',
       'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL',
       'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE',
       ...
       'POS_SK_DPD_DEF_max', 'POS_SK_DPD_DEF_min',
       'POS_NAME_CONTRACT_STATUS_NUNIQUE',
       'POS_NAME_CONTRACT_STATUS_Amortized debt',
       'POS_NAME_CONTRACT_STATUS_Approved',
       'POS_NAME_CONTRACT_STATUS_Canceled',
       'POS_NAME_CONTRACT_STATUS_Completed',
       'POS_NAME_CONTRACT_STATUS_Returned to the store',
       'POS_NAME_CONTRACT_STATUS_Signed', 'POS_NAME_CONTRACT_STATUS_XNA'],
      dtype='object', length=327)

In [ ]:
df_example['PREV_NAME_GOODS_CATEGORY_Direct Sales']

In [26]:
df_200 = df_example.sample(200, random_state=42)

In [ ]:
df_200['PREV_NAME_GOODS_CATEGORY_Direct Sales']